> This post introduces `helix` - a Claude Code orchestrator that learns from every session.

# Introduction

Before Opus 4.5, agentic harnesses focused on working *around* the two worst tendencies of LLMs: scope creep and over-engineering. Coding agents felt like overeager junior-savants that had to be carefully steered whenever projects became even moderately complex.

Opus 4.5 broke this pattern. If you've been using it, you've likely felt the shift. We are now living the transformation of LLM agents from spastic assistants to true collaborators.

`helix` is built on this shift. While previous harnesses constrained models to prevent drift, `helix` persists knowledge across sessions to build on what *worked* - not just what was done.

`helix` is the successor to [`ftl`](../011_ftl_orchestrator/ftl_orchestrator.ipynb), rebuilt around a single insight: memory without feedback is just storage.

This post goes over what it does and how it fits together.

# Philosophy

`helix` is built on six principles:

| Principle | What it means |
|-----------|---------------|
| **Feedback closes the loop** | Memories that help rise in ranking. Memories that don't help sink. |
| **Verify first** | Shape work by starting with proof-of-success |
| **Bounded scope** | Delta files are explicit so humans can audit agent boundaries |
| **Present over future** | Implement current requests, not anticipated needs |
| **Edit over create** | Modify what exists before creating something new |
| **Blocking is success** | Clear blocking info is better than broken code |

The word "feedback" in the first principle is load-bearing. In `ftl`, memory accumulated - each task left artifacts that persisted. But persistence is not learning. A filing cabinet that grows larger is not getting smarter.

`helix` closes the loop. When memory is injected into a task, we track whether it actually helped. Memories that consistently help rise to the top. Memories that get injected but ignored sink toward oblivion. This is where things get interesting.

# The Development Loop

```
/helix <objective>
    │
    ▼
┌─────────────────────────────────────┐
│  EXPLORER (haiku, 6 tools)          │
│  structure │ patterns │ memory │ targets
└─────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────┐
│  PLANNER (opus)                     │
│  Decompose → Dependencies → Budget  │
└─────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────┐
│  BUILDER (opus, budget 5-9)         │
│  Read → Implement → Verify → Report │
└─────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────┐
│  OBSERVER (opus)                    │
│  Extract failures │ Chunk patterns  │
└─────────────────────────────────────┘
```

Four specialized agents, each with a distinct role. The Explorer gathers context. The Planner decomposes objectives into tasks with dependencies. The Builder executes within strict constraints. The Observer extracts learning from outcomes.

Memory flows through the entire pipeline:

```
recall() → inject → feedback() → store()
```

The Explorer queries memory for relevant context before any planning happens. The Builder receives injected memories and reports which ones actually helped. The Observer stores new failures and patterns. The feedback loop adjusts rankings based on what worked.

Each completed objective makes the system smarter - but only if feedback is honest.

# Agents

`helix` coordinates four specialized agents:

| Agent | Model | Role | Budget |
|-------|-------|------|--------|
| **Explorer** | Haiku | Codebase reconnaissance | 6 |
| **Planner** | Opus | Task DAG decomposition | unlimited |
| **Builder** | Opus | Execution within constraints | 5-9 |
| **Observer** | Opus | Learning extraction | 10 |

The division of labor is deliberate.

**Explorer** runs on Haiku for cost efficiency. Its job is reconnaissance, not reasoning - find the structure, detect the framework, query memory for relevant context, identify target files. Six tool calls is enough for exploration. More would be scope creep.

**Planner** runs on Opus with no tool budget limit. Planning is the highest-leverage activity - a poor plan wastes every downstream tool call. The Planner decomposes complex objectives into focused tasks, sets dependencies (parallelizing where possible), and assigns budgets based on complexity.

**Builder** runs on Opus with a tight budget (5-9 tools per task). This is the constraint that prevents spiral. If a task can't be completed within budget, the Builder blocks with what it tried. The budget forces focus: read the delta files, implement the change, verify, report.

**Observer** runs on Opus with enough budget to analyze outcomes. Its job is extracting generalizable knowledge from what just happened - failures from blocked tasks, patterns from successful completions, relationships between memories.

# Task DAG

The Planner doesn't just create a list of tasks. It creates a directed acyclic graph with explicit dependencies.

```
001: spec-auth-models ─┬─→ 002: spec-auth-tests ─┬─→ 004: impl-auth-routes
                       │                         │
                       └─→ 003: impl-auth-service ─┘
```

Each task has:

| Field | Purpose |
|-------|--------|
| `seq` | Execution order identifier ("001", "002") |
| `slug` | Human-readable name ("spec-auth-models") |
| `objective` | What this task accomplishes |
| `delta` | Files this task may modify (strict constraint) |
| `verify` | Command to verify completion |
| `depends` | Tasks that must complete first ("none" or "001,002") |
| `budget` | Tool calls allocated (5-9) |

Tasks are registered with Claude Code's native task system, visible via `Ctrl+T` or `/todos`. This gives humans visibility into progress as the pipeline executes.

Tasks can run in parallel when they don't depend on each other. The DAG structure means `002` and `003` can execute simultaneously once `001` completes.

# Memory System

This is where `helix` diverges most from its predecessor. The memory system isn't just storage - it's a learning system with effectiveness tracking and decay.

## Storage

Everything lives in a single SQLite database at `.helix/helix.db`. No scattered JSON files, no complex file hierarchies. Memories are stored with 384-dimensional embeddings for semantic search - queries find relevant knowledge by meaning, not keywords.

| Table | Purpose |
|-------|--------|
| `memory` | Failures and patterns with embeddings |
| `memory_edge` | Relationships between memories |
| `exploration` | Gathered context from Explorer |
| `plan` | Task decompositions from Planner |
| `workspace` | Task execution contexts |

## Scoring Formula

When retrieving memories, ranking is not just semantic relevance. The scoring formula:

```
score = (0.5 × relevance) + (0.3 × effectiveness) + (0.2 × recency)
```

Where:
- **relevance** = cosine similarity between query and memory embeddings
- **effectiveness** = `helped / (helped + failed)`, default 0.5 if no feedback
- **recency** = `2^(-days_since_use / 7)` (ACT-R decay)

The ACT-R decay comes from cognitive architecture research. Memories that haven't been used recently fade, matching how human memory works. A memory that helped six months ago but hasn't been touched since will have lower recency than one used yesterday.

## The Feedback Loop

This is the critical function:

```python
feedback(utilized, injected)
# utilized memories: helped++
# injected-but-unused: failed++
```

When the Builder completes a task, it reports which memories were actually utilized. The feedback function compares this to what was injected:

- Memory was injected AND utilized? → `helped` counter increments
- Memory was injected but NOT utilized? → `failed` counter increments

Over time, effective memories rise in ranking. Memories that consistently get injected but ignored sink. The system learns which memories actually help - not which ones seemed relevant at query time.

In short: memory without feedback is storage. Memory with feedback is learning.

## SOAR Chunking

The Observer extracts patterns using SOAR-style chunking. When a task succeeds with a notable technique:

```bash
python3 $HELIX_PLUGIN_ROOT/lib/memory/core.py chunk \
    --task "What was accomplished" \
    --outcome "SUCCESS" \
    --approach "The technique that worked"
```

This captures the transition from deliberate problem-solving to compiled expertise. A technique that worked once becomes a retrievable pattern for similar future situations.

## Graph Relationships

Memories don't exist in isolation. The system tracks edges between them:

| Type | Meaning |
|------|--------|
| `co_occurs` | These failures tend to appear together |
| `causes` | This failure leads to that failure |
| `solves` | This pattern resolves that failure |
| `similar` | These memories are semantically close |

The `connected()` traversal lets you explore the neighborhood of a memory - if you hit failure A, what related failures should you watch for? What patterns have solved it before?

## Maintenance

The memory system requires occasional maintenance:

| Operation | What it does |
|-----------|-------------|
| `consolidate()` | Merge semantically similar memories |
| `prune()` | Remove memories with effectiveness < 0.25 |
| `decay()` | Find dormant memories that haven't been used |

# Commands

| Command | Purpose |
|---------|--------|
| `/helix <objective>` | Full pipeline: explore → plan → build → observe |
| `/helix-query "topic"` | Search memory by semantic similarity |
| `/helix-stats` | Memory health metrics and feedback loop status |

# Blocking as Learning

When a task goes sideways, the Builder has a hard constraint: 5-9 tools max. If it hasn't solved the problem within budget, it's exploring, not debugging. At that point, or after hitting the same error approach three times, the Builder blocks.

Blocking isn't failure. It's the system working correctly.

The workspace records what was tried:

```
BLOCKED: Need to modify src/main.py but it's not in delta
TRIED: Implemented auth service in src/services/auth.py
ERROR: Cannot import auth routes without modifying main.py
```

An agent that says "this is beyond what I can debug, here's what I tried" is more valuable than one that burns 100k tokens spiraling. The confidence to escalate is a feature.

The metacognition check is explicit: after three failed attempts with similar approaches, the Builder must stop and analyze rather than retry. Is there a fundamentally different approach? Is the task mis-scoped? Is information missing?

Blocked workspaces feed the Observer. Every block is a potential failure pattern to extract - a lesson for future tasks encountering similar situations.

# When to Use

**Use helix when:**

- Work should persist as learned knowledge that compounds over time
- Complex objectives need decomposition into verifiable tasks
- You want bounded, reviewable scope with explicit file constraints
- Past failures should inform future attempts
- Framework-specific development benefits from detected idioms

**Skip helix when:**

- Simple single-file changes that don't benefit from orchestration
- Exploratory prototyping where you want the model to wander
- Quick one-offs with no future value

Knowing when to reach for these tools - and when not to - is itself a skill.

# Installation

```bash
# Add the crinzo-plugins marketplace
claude plugin marketplace add https://github.com/enzokro/crinzo-plugins

# Install helix
claude plugin install helix@crinzo-plugins
```

Or from inside Claude Code:

```bash
/plugin marketplace add https://github.com/enzokro/crinzo-plugins
/plugin install helix@crinzo-plugins
```

# Conclusion

Opus 4.5 proved that agents can be true collaborators. What was missing was the architecture to let that collaboration compound.

`helix` builds on `ftl` with a crucial addition: feedback. Memory accumulation is not learning. A system that tracks which memories actually helped - and adjusts rankings based on that signal - is a system that improves over time.

The core insight remains: context loss is an architecture problem, not a capability problem. If we structure knowledge to persist, it will persist. If we track what works, the system learns. If we create explicit boundaries, scope stays contained.

Memory without feedback is storage. Memory with feedback is learning.

The models are ready. The scaffolding is getting smarter.